In [ ]:
!pip install -q transformers accelerate fastapi uvicorn pyngrok

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.responses import StreamingResponse
import torch
import uvicorn
import threading
import json

app = FastAPI()


class ChatRequest(BaseModel):
    model: str
    messages: list
    stream: bool = False
    temperature: float = 0.7
    max_tokens: int = 200


@app.post("/v1/chat/completions")
def chat_completions(request: ChatRequest):

    # Get user message
    user_prompt = ""

    for message in request.messages:
        if message["role"] == "user":
            user_prompt = message["content"]

    messages = [
        {"role": "user", "content": user_prompt}
    ]

    # Qwen chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    # Generate
    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=request.max_tokens,
            temperature=request.temperature,
            do_sample=True
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    # -----------------------------
    # STREAMING RESPONSE
    # -----------------------------

    if request.stream:

        def stream_response():

            chunk = {
                "id": "qwen-local",
                "object": "chat.completion.chunk",
                "model": request.model,
                "choices": [
                    {
                        "index": 0,
                        "delta": {
                            "role": "assistant",
                            "content": response
                        },
                        "finish_reason": None
                    }
                ]
            }

            yield "data: " + json.dumps(chunk) + "\n\n"

            end_chunk = {
                "id": "qwen-local",
                "object": "chat.completion.chunk",
                "model": request.model,
                "choices": [
                    {
                        "index": 0,
                        "delta": {},
                        "finish_reason": "stop"
                    }
                ]
            }

            yield "data: " + json.dumps(end_chunk) + "\n\n"

            yield "data: [DONE]\n\n"

        return StreamingResponse(
            stream_response(),
            media_type="text/event-stream"
        )

    # -----------------------------
    # NORMAL RESPONSE
    # -----------------------------

    return {
        "id": "qwen-local",
        "object": "chat.completion",
        "model": request.model,
        "choices": [
            {
                "index": 0,
                "message": {
                    "role": "assistant",
                    "content": response
                },
                "finish_reason": "stop"
            }
        ]
    }


@app.get("/v1/models")
def list_models():

    return {
        "object": "list",
        "data": [
            {
                "id": "Qwen/Qwen2.5-1.5B-Instruct",
                "object": "model",
                "owned_by": "local"
            }
        ]
    }


print("FastAPI app created!")

FastAPI app created!


In [ ]:
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)


thread = threading.Thread(
    target=run_server,
    daemon=True
)

thread.start()

print("FastAPI server running on port 8000")

FastAPI server running on port 8000


In [ ]:
!ngrok config add-authtoken 3HrLXFno4VWR9DwV6XnSRR0QCB0_4UtWpypj1CNdgMjJjaM66

INFO:     Started server process [612]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8000)

print("NEW PUBLIC URL:")
print(public_url)

NEW PUBLIC URL:
NgrokTunnel: "https://java-aloha-napped.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
from google.colab import files

# To download the ngrok.yml file
files.download('/root/.config/ngrok/ngrok.yml')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests

url = "https://java-aloha-napped.ngrok-free.dev/v1/models"

response = requests.get(url)

print("Status:", response.status_code)
print("Response:", response.text)

INFO:     35.240.132.112:0 - "GET /v1/models HTTP/1.1" 200 OK
Status: 200
Response: {"object":"list","data":[{"id":"Qwen/Qwen2.5-1.5B-Instruct","object":"model","owned_by":"local"}]}


In [ ]:
import requests

url = "https://java-aloha-napped.ngrok-free.dev/v1/chat/completions"

payload = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "Hello, introduce yourself in one sentence."
        }
    ],
    "temperature": 0.7,
    "max_tokens": 100,
    "stream": False
}

response = requests.post(url, json=payload)

print("Status:", response.status_code)
print("Response:", response.text)

INFO:     35.240.132.112:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Status: 200
Response: {"id":"qwen-local","object":"chat.completion","model":"Qwen/Qwen2.5-1.5B-Instruct","choices":[{"index":0,"message":{"role":"assistant","content":"I am an artificial intelligence designed to assist with various tasks and answer questions to the best of my ability based on the information provided."},"finish_reason":"stop"}]}


In [ ]:
import requests

url = "https://java-aloha-napped.ngrok-free.dev/v1/chat/completions"

payload = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "Say hello in one sentence."
        }
    ],
    "stream": True,
    "temperature": 0.7,
    "max_tokens": 50
}

response = requests.post(
    url,
    json=payload,
    stream=True
)

print("Status:", response.status_code)

for line in response.iter_lines():
    if line:
        print(line.decode())

INFO:     35.240.132.112:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Status: 200
data: {"id": "qwen-local", "object": "chat.completion.chunk", "model": "Qwen/Qwen2.5-1.5B-Instruct", "choices": [{"index": 0, "delta": {"role": "assistant", "content": "Hello! How can I assist you today?"}, "finish_reason": null}]}
data: {"id": "qwen-local", "object": "chat.completion.chunk", "model": "Qwen/Qwen2.5-1.5B-Instruct", "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}]}
data: [DONE]


In [ ]:
import requests

url = "https://java-aloha-napped.ngrok-free.dev/v1/chat/completions"

payload = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "Say exactly HELLO QWEN"
        }
    ],
    "stream": True,
    "temperature": 0.7,
    "max_tokens": 50
}

response = requests.post(
    url,
    json=payload,
    stream=True
)

print("STATUS:", response.status_code)
print("CONTENT TYPE:", response.headers.get("content-type"))
print("RAW RESPONSE:")

for line in response.iter_lines():
    if line:
        print(line.decode())

INFO:     35.240.132.112:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
STATUS: 200
CONTENT TYPE: text/event-stream; charset=utf-8
RAW RESPONSE:
data: {"id": "qwen-local", "object": "chat.completion.chunk", "model": "Qwen/Qwen2.5-1.5B-Instruct", "choices": [{"index": 0, "delta": {"role": "assistant", "content": "Hello QWEN! How can I assist you today?"}, "finish_reason": null}]}
data: {"id": "qwen-local", "object": "chat.completion.chunk", "model": "Qwen/Qwen2.5-1.5B-Instruct", "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}]}
data: [DONE]


In [ ]:
import requests

url = "https://java-aloha-napped.ngrok-free.dev/v1/chat/completions"

payload = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "Say exactly HELLO QWEN"
        }
    ],
    "stream": True,
    "temperature": 0.7,
    "max_tokens": 50
}

response = requests.post(
    url,
    json=payload,
    stream=True
)

print("STATUS:", response.status_code)
print("CONTENT TYPE:", response.headers.get("content-type"))
print("RAW RESPONSE:")

for line in response.iter_lines():
    if line:
        print(line.decode())

INFO:     35.240.132.112:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
STATUS: 200
CONTENT TYPE: text/event-stream; charset=utf-8
RAW RESPONSE:
data: {"id": "qwen-local", "object": "chat.completion.chunk", "model": "Qwen/Qwen2.5-1.5B-Instruct", "choices": [{"index": 0, "delta": {"role": "assistant", "content": "Hello QWEN! How can I assist you today?"}, "finish_reason": null}]}
data: {"id": "qwen-local", "object": "chat.completion.chunk", "model": "Qwen/Qwen2.5-1.5B-Instruct", "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}]}
data: [DONE]
